In [ ]:
import pandas as pd
from dask.array import average
from gototile.grid import SkyGrid
from goto_sim.scheduling import HEATSurvey
from goto_sim.utils import load_tilelist
from astropy.time import Time, TimeDelta
import numpy as np
import matplotlib.pyplot as plt
import glob

In [ ]:
data_dir = "sims"
file_pattern = f"{data_dir}/simulation_results_reserve_*.csv"
file_list = glob.glob(file_pattern)
print(f"Found {len(file_list)} files matching the pattern.")
reserve_fractions = [float(x .split('_')[-1].split('.csv')[0]) for x in file_list]
results = {}
for file_path, reserve_fraction in zip(file_list, reserve_fractions):
    df = pd.read_csv(file_path, comment='#')
    results[reserve_fraction] = df
results_df = pd.concat(results, names=['reserve_fraction', 'index']).reset_index(level='reserve_fraction')

heats_survey = HEATSurvey()
heats_tiles = heats_survey.tiles
cold_survey = heats_survey.generate_colds()
cold_tiles = cold_survey.get_tiles()
grid: SkyGrid = SkyGrid.from_name("GOTO")


In [ ]:
cadence = results_df.groupby(['tile', 'reserve_fraction'])['time'].apply(
    lambda x: np.median(np.diff(np.sort(x)))
).sort_index()
print(f"Total results: {len(cadence)}")
cold_cadences = cadence.loc[cold_tiles]
print(f"COLD results: {len(cold_cadences)}")

heats_cadences = cadence.loc[heats_tiles]
print(f"HEATS results: {len(heats_cadences)}")


In [ ]:
average_cold_cadence = cold_cadences.groupby('reserve_fraction').mean()
average_heats_cadence = heats_cadences.groupby('reserve_fraction').mean()
plt.figure(figsize=(10, 6))
plt.plot(average_cold_cadence.index/2, average_cold_cadence.values, label='COLD Survey', marker='o', color='tab:blue')
plt.plot(average_heats_cadence.index/2, average_heats_cadence.values, label='HEATS Survey', marker='o', color='tab:red')
plt.xlabel('Reserved Time')
plt.ylabel('Median Cadence (Days)')
plt.title('Average Cadence vs Reserve Fraction for COLD and HEATS Surveys')
plt.yticks(minor=True, ticks=np.arange(0, max(average_heats_cadence.max(), average_cold_cadence.max()) + 5, 1))
plt.ylim(0, np.floor(max(average_heats_cadence.max(), average_cold_cadence.max()) + 2))
plt.legend()
plt.grid()
plt.show()